# EDA SECOP-DANE 2018-2024 -- Análisis Territorial Consolidado

**Solo la parte territorial** (desagregación por departamento y región) de las secciones:

| Sección | Tema |
|---------|------|
| S1 | Carga de datos e inspección por departamento y región |
| S6 | Gini intra-departamental e intra-regional |
| S7 | Correlación población vs IPC por departamento y región |
| S9 | Cuadrantes de inversión con medianas propias de cada territorio |
| S13 | % del presupuesto hacia municipios de baja IPC por depto y región |

> Ejecutar desde la raíz del proyecto (donde existe `data/gold/marts/latest/`).

## 0. Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Detección robusta de la raíz del proyecto ──────────────────────
_here = Path().resolve()
ROOT = _here
for _c in [_here, _here.parent, _here.parent.parent, _here.parent.parent.parent]:
    if (_c / 'data' / 'gold' / 'marts' / 'latest').exists():
        ROOT = _c
        break

MART_PATH      = ROOT / 'data' / 'gold' / 'marts' / 'latest' / 'mart_desarrollo_social_economico_municipio_anio.parquet'
CNPV_5PER_PATH = ROOT / 'data' / 'bronze' / 'cnpv' / 'cnpv_5per_raw.parquet'

# ── Constantes ─────────────────────────────────────────────────────
AÑOS           = list(range(2018, 2025))
COL_AÑO        = 'anio_key'
COL_NBI        = 'nbi_pct'
COL_IPM        = 'ipm_total'
COL_MONTO      = 'inversion_total_monto'
COL_CONTRATOS  = 'cantidad_procesos_adjudicados'
COL_DIVIPOLA   = 'divipola_key'
COL_MUNICIPIO  = 'nombre_municipio_referencia'
COL_DEPTO      = 'nombre_departamento'
COL_IPC        = 'indicador_inversion_per_capita'
COD_BOGOTA     = '11001'

# ── Estilo ─────────────────────────────────────────────────────────
plt.style.use('ggplot')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12,
                     'axes.labelsize': 10, 'font.size': 9})
PALETA_AÑOS = plt.cm.tab10(np.linspace(0, 0.9, len(AÑOS)))
COLOR_MAP   = dict(zip(AÑOS, PALETA_AÑOS))

OUT = ROOT / 'output_territorial'
OUT.mkdir(exist_ok=True)

def fmt_moneda(x, pos=None):
    if pd.isna(x): return 'N/A'
    if x >= 1e12: return f'${x/1e12:.1f}T'
    if x >= 1e9:  return f'${x/1e9:.1f}B'
    if x >= 1e6:  return f'${x/1e6:.1f}M'
    if x >= 1e3:  return f'${x/1e3:.0f}K'
    return f'${x:,.0f}'

def gini(array):
    a = np.array(array, dtype=np.float64)
    a = np.sort(a[(~np.isnan(a)) & (a > 0)])
    if len(a) == 0 or a.sum() == 0: return np.nan
    n = len(a)
    return (n + 1 - 2 * np.sum(np.cumsum(a)) / a.sum()) / n

print(f'Raiz del proyecto  : {ROOT}')
print(f'Mart existe        : {MART_PATH.exists()}')
print(f'CNPV bronze existe : {CNPV_5PER_PATH.exists()}')
print(f'Output graficas    : {OUT}')


## S1. Carga de datos e inspección territorial

Partimos del **Gold Mart** como fuente principal. Se enriquece con **etnia** calculada on-the-fly desde el bronze CNPV 2018 (`PA1_GRP_ETNIC`).

> **NBI / IPM:** no disponibles en el pipeline actual. Las secciones que antes dependían de estas variables usan ahora `indicador_inversion_per_capita` como proxy de equidad territorial.

### 1.1 Carga del Gold Mart y enriquecimiento con etnia CNPV

In [ ]:
# ── Gold Mart ───────────────────────────────────────────────────────
mart = pd.read_parquet(MART_PATH)
mart[COL_AÑO] = pd.to_numeric(mart[COL_AÑO], errors='coerce').astype('Int64')
mart = mart[mart[COL_AÑO].isin(AÑOS)].copy()
mart[COL_DIVIPOLA] = mart[COL_DIVIPOLA].astype(str).str.strip().str.zfill(5)

# ── Etnia (CNPV 2018 -- calculada desde bronze cnpv_5per_raw) ────────
# PA1_GRP_ETNIC: 1=Indígena 2=Gitano/ROM 3=Raizal 4=Palenquero 5=Afrodescendiente 6=Ninguno
if CNPV_5PER_PATH.exists():
    print("Calculando etnia desde CNPV bronze (lectura columnar, ~30 s)...")
    _e = pd.read_parquet(CNPV_5PER_PATH, columns=['U_DPTO', 'U_MPIO', 'PA1_GRP_ETNIC'])
    _e['divipola_key'] = (
        _e['U_DPTO'].astype(str).str.strip().str.zfill(2) +
        _e['U_MPIO'].astype(str).str.strip().str.zfill(3))
    _e = _e[_e['divipola_key'].str.match(r'^\d{5}$')]
    _tot = _e.groupby('divipola_key').size().rename('total')
    _ind = _e[_e['PA1_GRP_ETNIC'] == 1].groupby('divipola_key').size().rename('ind')
    _afr = _e[_e['PA1_GRP_ETNIC'] == 5].groupby('divipola_key').size().rename('afro')
    etnia_sub = pd.concat([_tot, _ind, _afr], axis=1).fillna(0).reset_index()
    etnia_sub['etnia_indigena_pct'] = etnia_sub['ind']  / etnia_sub['total'] * 100
    etnia_sub['etnia_afro_pct']     = etnia_sub['afro'] / etnia_sub['total'] * 100
    etnia_sub = etnia_sub[['divipola_key', 'etnia_indigena_pct', 'etnia_afro_pct']]
    print(f"Etnia calculada para {len(etnia_sub):,} municipios.")
else:
    print("ADVERTENCIA: CNPV bronze no encontrado. Etnia no disponible.")
    etnia_sub = pd.DataFrame(columns=['divipola_key', 'etnia_indigena_pct', 'etnia_afro_pct'])

# ── NBI / IPM ────────────────────────────────────────────────────────
sprint2_vuln = pd.DataFrame(columns=['divipola_key', COL_NBI, 'miseria_pct', COL_IPM])

# ── Enriquecimiento del Mart ─────────────────────────────────────────
df = (mart
      .merge(sprint2_vuln, on='divipola_key', how='left')
      .merge(etnia_sub,    on='divipola_key', how='left'))

# Solo municipios (excluir agregados departamentales XX000)
df_mun = df[~df['divipola_key'].str.endswith('000')].copy()

# Tipos numéricos
_num_cols = [COL_MONTO, COL_CONTRATOS, 'proveedores_unicos',
             'poblacion_total_proyectada', 'poblacion_censo_2018',
             'volumen_micronegocios_exp', COL_NBI, 'miseria_pct', COL_IPM,
             'indicador_inversion_per_capita', 'etnia_indigena_pct', 'etnia_afro_pct']
for col in _num_cols:
    if col in df_mun.columns:
        df_mun[col] = pd.to_numeric(df_mun[col], errors='coerce')

COL_POB = 'poblacion_censo_2018'

print('=' * 65)
print('INSPECCION INICIAL -- Gold Mart + Etnia CNPV 2018')
print('=' * 65)
print(f'Dimensiones totales     : {df.shape[0]:,} x {df.shape[1]}')
print(f'Solo municipios         : {df_mun.shape[0]:,} x {df_mun.shape[1]}')
print(f'\nMunicipios unicos  : {df_mun[COL_DIVIPOLA].nunique():,}')
print(f'Departamentos      : {df_mun[COL_DEPTO].nunique():,}')
print(f'Regiones           : {sorted(df_mun["region"].dropna().unique())}')
print(f'Columna poblacion  : {COL_POB}')


### 1.2 Inspección por departamento

Resumen acumulado 2018-2024: municipios, monto total, contratos, IPC mediano, composición étnica.

In [ ]:
resumen_depto = df_mun.groupby(COL_DEPTO).agg(
    municipios    = (COL_DIVIPOLA, 'nunique'),
    monto_total   = (COL_MONTO,    'sum'),
    contratos     = (COL_CONTRATOS, 'sum'),
    ipc_mediana   = (COL_IPC,      'median'),
    pob_total     = (COL_POB,      'sum'),
    pct_indigena  = ('etnia_indigena_pct', 'mean'),
    pct_afro      = ('etnia_afro_pct',     'mean'),
    region        = ('region',      'first'),
).reset_index()
resumen_depto['monto_per_capita'] = resumen_depto['monto_total'] / resumen_depto['pob_total']
resumen_depto = resumen_depto.sort_values('monto_total', ascending=False)

print('\nTop 10 departamentos por monto total acumulado:')
cols_d = ['nombre_departamento', 'municipios', 'monto_total', 'ipc_mediana', 'region']
top10 = resumen_depto.head(10)
for _, r in top10.iterrows():
    print(f"  {r['nombre_departamento']:<20s}  {fmt_moneda(r['monto_total']):>8s}"
          f"  ({int(r['municipios'])} mun.)  IPC med: {fmt_moneda(r['ipc_mediana'])}"
          f"  [{r['region']}]")

# Gráfica: barras de monto por departamento
fig, ax = plt.subplots(figsize=(12, 16))
orden = resumen_depto.sort_values('monto_total')['nombre_departamento'].tolist()
valores = resumen_depto.sort_values('monto_total')['monto_total'].tolist()
ax.barh(range(len(orden)), valores,
        color=plt.cm.Blues(np.linspace(0.4, 0.9, len(orden))),
        edgecolor='black', alpha=0.85)
ax.set_yticks(range(len(orden)))
ax.set_yticklabels(orden, fontsize=6.5); ax.invert_yaxis()
ax.set_title('Monto Total Contratado por Departamento (2018-2024)', fontweight='bold')
ax.set_xlabel('Monto total ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_moneda))
plt.tight_layout()
plt.savefig(OUT / 'S1_monto_por_depto.png', dpi=150, bbox_inches='tight')
plt.show()


### 1.3 Inspección por región

In [ ]:
resumen_region = df_mun.groupby('region').agg(
    municipios    = (COL_DIVIPOLA, 'nunique'),
    deptos        = (COL_DEPTO,    'nunique'),
    monto_total   = (COL_MONTO,    'sum'),
    contratos     = (COL_CONTRATOS, 'sum'),
    ipc_mediana   = (COL_IPC,      'median'),
    pob_total     = (COL_POB,      'sum'),
    pct_indigena  = ('etnia_indigena_pct', 'mean'),
    pct_afro      = ('etnia_afro_pct',     'mean'),
).reset_index()
resumen_region['monto_per_capita'] = resumen_region['monto_total'] / resumen_region['pob_total']
resumen_region = resumen_region.sort_values('monto_total', ascending=False)

print('\nResumen por region:')
for _, r in resumen_region.iterrows():
    print(f"  {r['region']:<15s}  {int(r['municipios']):>4d} mun.  {int(r['deptos']):>2d} deptos."
          f"  Monto: {fmt_moneda(r['monto_total']):>8s}  IPC med: {fmt_moneda(r['ipc_mediana'])}")

# Gráfica: composición regional
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax = axes[0]
montos_r = resumen_region.sort_values('region')['monto_total'].values
labels_r = resumen_region.sort_values('region')['region'].values
ax.pie(montos_r, labels=labels_r, autopct='%1.1f%%',
       colors=plt.cm.Set3(np.linspace(0, 1, len(labels_r))),
       explode=[0.03]*len(labels_r), startangle=90,
       textprops={'fontsize': 7})
ax.set_title('% del Monto Total por Region', fontweight='bold')
ax = axes[1]
reg_ord = resumen_region.sort_values('municipios', ascending=False)
ax.bar(range(len(reg_ord)), reg_ord['municipios'].values,
       color=plt.cm.Set3(np.linspace(0, 1, len(reg_ord))), edgecolor='black')
ax.set_xticks(range(len(reg_ord)))
ax.set_xticklabels(reg_ord['region'].values, rotation=30, ha='right', fontsize=8)
ax.set_title('Municipios por Region', fontweight='bold')
ax.set_ylabel('Numero de municipios')
for i, v in enumerate(reg_ord['municipios'].values):
    ax.text(i, v + 3, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUT / 'S1_composicion_region.png', dpi=150, bbox_inches='tight')
plt.show()


## S6. Gini intra-departamental e intra-regional

Mide la **desigualdad dentro de cada territorio**. Responde dos preguntas distintas:

- **(a) Gini de monto absoluto** -- concentración geográfica del gasto (descriptivo, NO indicador de equidad)
- **(b) Gini de inversión per cápita (IPC)** -- equidad territorial (MÉTRICA PRINCIPAL)

**Reglas metodológicas:**
- Solo municipios con monto > 0 (no se imputan ceros)
- Para IPC se requiere monto > 0 y población > 0
- Se excluye Bogotá D.C. (`11001`) como proxy de "sin orden nacional"
- Cada departamento/región usa sus PROPIOS municipios para el cálculo

### 6.1 Gini por departamento

In [ ]:
print("Calculando Gini intra-departamental para cada año...")
filas_gini_depto = []
for a in AÑOS:
    sub = df_mun[df_mun[COL_AÑO] == a]
    for depto, grp in sub.groupby(COL_DEPTO):
        # (a) Gini monto absoluto
        m_depto = grp.loc[grp[COL_MONTO] > 0, COL_MONTO].values
        g_abs = gini(m_depto)
        # (a') sin Bogotá
        grp_nb = grp[grp[COL_DIVIPOLA] != COD_BOGOTA]
        m_nb = grp_nb.loc[grp_nb[COL_MONTO] > 0, COL_MONTO].values
        g_abs_nb = gini(m_nb)
        # (b) Gini IPC
        grp_ipc = grp[(grp[COL_MONTO] > 0) & (grp[COL_POB] > 0)].copy()
        g_ipc = np.nan; g_ipc_nb = np.nan
        if len(grp_ipc) >= 3:
            grp_ipc['ipc'] = grp_ipc[COL_MONTO] / grp_ipc[COL_POB]
            g_ipc = gini(grp_ipc['ipc'].values)
            vals_nb = grp_ipc[grp_ipc[COL_DIVIPOLA] != COD_BOGOTA]['ipc'].values
            if len(vals_nb) >= 3:
                g_ipc_nb = gini(vals_nb)
        filas_gini_depto.append({
            'año': int(a), 'departamento': depto,
            'region': grp['region'].iloc[0] if 'region' in grp.columns else 'N/A',
            'gini_monto_abs': g_abs, 'gini_monto_sin_bog': g_abs_nb,
            'gini_ipc': g_ipc, 'gini_ipc_sin_bog': g_ipc_nb,
            'n_muni': len(m_depto), 'monto_total': grp[COL_MONTO].sum(),
        })

gini_depto = pd.DataFrame(filas_gini_depto)

# Promedio 2018-2024 por departamento
gini_depto_prom = (gini_depto
    .groupby('departamento')
    .agg(gini_ipc_mean   = ('gini_ipc_sin_bog', 'mean'),
         gini_monto_mean = ('gini_monto_abs',   'mean'),
         n_muni          = ('n_muni',           'first'),
         region          = ('region',           'first'),
         monto_total     = ('monto_total',      'sum'))
    .sort_values('gini_ipc_mean', ascending=False)
    .reset_index())

print(f"\nRegistros departamento-año: {len(gini_depto)}")
print(f"Departamentos analizados: {gini_depto['departamento'].nunique()}")

print("\nTop 5 deptos con MAYOR desigualdad intra-departamental (Gini IPC sin Bogota):")
for _, r in gini_depto_prom.head(5).iterrows():
    print(f"  {r['departamento']:<20s}  Gini IPC = {r['gini_ipc_mean']:.3f}"
          f"  ({int(r['n_muni'])} mun.)  [{r['region']}]")

print("\nTop 5 deptos con MENOR desigualdad (mas equitativos):")
for _, r in gini_depto_prom.tail(5).iloc[::-1].iterrows():
    print(f"  {r['departamento']:<20s}  Gini IPC = {r['gini_ipc_mean']:.3f}"
          f"  ({int(r['n_muni'])} mun.)  [{r['region']}]")

# Gráfica: top/bottom Gini IPC
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.suptitle('Gini de Inversion per Capita por Departamento\n'
             '(promedio 2018-2024, sin Bogota D.C.)', fontweight='bold')
ax = axes[0]
top = gini_depto_prom.head(10).iloc[::-1]
ax.barh(range(10), top['gini_ipc_mean'],
        color=plt.cm.Reds(np.linspace(0.5, 0.9, 10)), edgecolor='black')
ax.set_yticks(range(10)); ax.set_yticklabels(top['departamento']); ax.set_xlim(0, 1)
ax.set_title('Mayor desigualdad intra-depto', fontweight='bold')
ax.set_xlabel('Gini IPC (sin Bogota)')
for i, (v, n) in enumerate(zip(top['gini_ipc_mean'], top['n_muni'])):
    ax.text(v + 0.01, i, f'{v:.3f}  ({int(n)} mun.)', va='center', fontsize=8)
ax = axes[1]
bot = gini_depto_prom.dropna(subset=['gini_ipc_mean']).tail(10).iloc[::-1]
ax.barh(range(len(bot)), bot['gini_ipc_mean'],
        color=plt.cm.Greens(np.linspace(0.5, 0.9, len(bot))), edgecolor='black')
ax.set_yticks(range(len(bot))); ax.set_yticklabels(bot['departamento']); ax.set_xlim(0, 1)
ax.set_title('Menor desigualdad (mas equitativos)', fontweight='bold')
ax.set_xlabel('Gini IPC (sin Bogota)')
for i, (v, n) in enumerate(zip(bot['gini_ipc_mean'], bot['n_muni'])):
    ax.text(v + 0.01, i, f'{v:.3f}  ({int(n)} mun.)', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(OUT / 'S6_gini_depto_topbottom.png', dpi=150, bbox_inches='tight')
plt.show()


### 6.2 Gini por región

In [ ]:
print("Calculando Gini intra-regional para cada año...")
filas_gini_reg = []
for a in AÑOS:
    sub = df_mun[df_mun[COL_AÑO] == a]
    for region, grp in sub.groupby('region'):
        if pd.isna(region) or region == '': continue
        grp_ipc = grp[(grp[COL_MONTO] > 0) & (grp[COL_POB] > 0)].copy()
        g_ipc_r = np.nan
        if len(grp_ipc) >= 3:
            grp_ipc['ipc'] = grp_ipc[COL_MONTO] / grp_ipc[COL_POB]
            g_ipc_r = gini(grp_ipc['ipc'].values)
        m_reg = grp.loc[grp[COL_MONTO] > 0, COL_MONTO].values
        filas_gini_reg.append({
            'año': int(a), 'region': region,
            'gini_ipc': g_ipc_r, 'gini_monto': gini(m_reg),
            'n_muni': len(m_reg), 'monto_total': grp[COL_MONTO].sum(),
        })

gini_region = pd.DataFrame(filas_gini_reg)
gini_reg_prom = (gini_region
    .groupby('region')
    .agg(gini_ipc_mean = ('gini_ipc', 'mean'),
         n_muni = ('n_muni', 'first'), monto_total = ('monto_total', 'sum'))
    .sort_values('gini_ipc_mean', ascending=False)
    .reset_index())

print("\nGini IPC por region (promedio 2018-2024):")
for _, r in gini_reg_prom.iterrows():
    print(f"  {r['region']:<15s}  Gini IPC = {r['gini_ipc_mean']:.3f}  ({int(r['n_muni'])} mun.)")

# Gráfica
fig, ax = plt.subplots(figsize=(10, 5))
colores = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(gini_reg_prom)))
bars = ax.barh(range(len(gini_reg_prom)), gini_reg_prom['gini_ipc_mean'],
               color=colores, edgecolor='black')
ax.set_yticks(range(len(gini_reg_prom)))
ax.set_yticklabels(gini_reg_prom['region']); ax.invert_yaxis(); ax.set_xlim(0, 1)
ax.set_title('Gini de IPC por Region (promedio 2018-2024)', fontweight='bold')
ax.set_xlabel('Gini IPC'); ax.axvline(0.5, color='gray', ls=':', lw=1.2)
for bar, v in zip(bars, gini_reg_prom['gini_ipc_mean']):
    ax.text(v + 0.01, bar.get_y() + bar.get_height()/2, f'{v:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig(OUT / 'S6_gini_region.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nLectura:')
print('  - Equidad territorial -> Gini IPC (mas bajo = mas equitativo)')
print('  - Concentracion geografica -> Gini monto (descriptivo, NO indicador de equidad)')


## S7. Correlación población vs IPC por departamento y región

Detecta si **dentro de cada territorio** los municipios más pequeños reciben proporcionalmente más o menos inversión que los grandes.

- **Eje X**: log₁₀(población censal 2018)
- **Eje Y**: inversión per cápita
- **r negativo** = municipios grandes reciben menos por persona
- **r positivo** = municipios grandes reciben más por persona

### 7.1 Correlación por departamento

In [ ]:
print("Calculando correlacion Pearson intra-departamental para cada año...")
filas_corr_depto = []
for a in AÑOS:
    sub = df_mun[
        (df_mun[COL_AÑO] == a) &
        (df_mun[COL_IPC] > 0) &
        (df_mun[COL_POB] > 0)
    ].dropna(subset=[COL_IPC, COL_POB]).copy()
    sub['log_pob'] = np.log10(sub[COL_POB])
    for depto, grp in sub.groupby(COL_DEPTO):
        if len(grp) < 5: continue
        r = grp['log_pob'].corr(grp[COL_IPC])
        filas_corr_depto.append({
            'año': int(a), 'departamento': depto,
            'region': grp['region'].iloc[0] if 'region' in grp.columns else 'N/A',
            'r_logPob_IPC': round(r, 4), 'n': len(grp),
        })

corr_depto = pd.DataFrame(filas_corr_depto)
corr_depto_prom = (corr_depto
    .groupby('departamento')
    .agg(r_mean = ('r_logPob_IPC', 'mean'), r_std = ('r_logPob_IPC', 'std'),
         n = ('n', 'first'), region = ('region', 'first'))
    .sort_values('r_mean')
    .reset_index())

print(f"\nRegistros departamento-año: {len(corr_depto)}")
print(f"Departamentos analizados: {corr_depto['departamento'].nunique()}")

print("\nTop 5 deptos con r MAS NEGATIVO (grandes reciben menos por persona):")
for _, r in corr_depto_prom.head(5).iterrows():
    print(f"  {r['departamento']:<20s}  r = {r['r_mean']:.4f}  ({int(r['n'])} mun.)  [{r['region']}]")

print("\nTop 5 deptos con r MAS POSITIVO (grandes reciben mas por persona):")
for _, r in corr_depto_prom.tail(5).iloc[::-1].iterrows():
    print(f"  {r['departamento']:<20s}  r = {r['r_mean']:.4f}  ({int(r['n'])} mun.)  [{r['region']}]")

# Gráfica
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.suptitle('Correlacion log10(Poblacion) vs IPC por Departamento\n'
             '(promedio 2018-2024)', fontweight='bold')
ax = axes[0]
top_neg = corr_depto_prom.head(10).iloc[::-1]
ax.barh(range(10), top_neg['r_mean'],
        color=plt.cm.Reds(np.linspace(0.5, 0.9, 10)), edgecolor='black',
        xerr=top_neg['r_std'])
ax.set_yticks(range(10)); ax.set_yticklabels(top_neg['departamento'])
ax.axvline(0, color='black')
ax.set_title('r mas NEGATIVO (sesgo anti-grandes)', fontweight='bold')
ax.set_xlabel('Pearson r')
for i, (v, sd, n) in enumerate(zip(top_neg['r_mean'], top_neg['r_std'], top_neg['n'])):
    ax.text(min(v, -0.01) - 0.02, i, f'{v:.3f}  (n~{int(n)})', ha='right', va='center', fontsize=8)
ax = axes[1]
top_pos = corr_depto_prom.tail(10).iloc[::-1]
ax.barh(range(10), top_pos['r_mean'],
        color=plt.cm.Blues(np.linspace(0.5, 0.9, 10)), edgecolor='black',
        xerr=top_pos['r_std'])
ax.set_yticks(range(10)); ax.set_yticklabels(top_pos['departamento'])
ax.axvline(0, color='black')
ax.set_title('r mas POSITIVO (sesgo pro-grandes)', fontweight='bold')
ax.set_xlabel('Pearson r')
for i, (v, sd, n) in enumerate(zip(top_pos['r_mean'], top_pos['r_std'], top_pos['n'])):
    ax.text(max(v, 0.01) + 0.02, i, f'{v:.3f}  (n~{int(n)})', ha='left', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(OUT / 'S7_corr_depto_topbottom.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.2 Correlación por región

In [ ]:
print("Calculando correlacion intra-regional...")
filas_corr_reg = []
for a in AÑOS:
    sub = df_mun[
        (df_mun[COL_AÑO] == a) &
        (df_mun[COL_IPC] > 0) &
        (df_mun[COL_POB] > 0)
    ].dropna(subset=[COL_IPC, COL_POB]).copy()
    sub['log_pob'] = np.log10(sub[COL_POB])
    for region, grp in sub.groupby('region'):
        if pd.isna(region) or region == '' or len(grp) < 5: continue
        r = grp['log_pob'].corr(grp[COL_IPC])
        filas_corr_reg.append({
            'año': int(a), 'region': region,
            'r_logPob_IPC': round(r, 4), 'n': len(grp),
        })

corr_region = pd.DataFrame(filas_corr_reg)
corr_reg_prom = (corr_region
    .groupby('region')
    .agg(r_mean = ('r_logPob_IPC', 'mean'), r_std = ('r_logPob_IPC', 'std'),
         n = ('n', 'first'))
    .sort_values('r_mean')
    .reset_index())

print("\nCorrelacion por region (promedio 2018-2024):")
for _, r in corr_reg_prom.iterrows():
    print(f"  {r['region']:<15s}  r = {r['r_mean']:.4f}  ({int(r['n'])} mun.)")

# Gráfica
fig, ax = plt.subplots(figsize=(10, 5))
colores = ['#E74C3C' if v < 0 else '#2980B9' for v in corr_reg_prom['r_mean']]
ax.barh(range(len(corr_reg_prom)), corr_reg_prom['r_mean'], color=colores,
        edgecolor='black', xerr=corr_reg_prom['r_std'], capsize=3)
ax.set_yticks(range(len(corr_reg_prom)))
ax.set_yticklabels(corr_reg_prom['region']); ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=1.2)
ax.set_title('Correlacion Poblacion vs IPC por Region\n'
             '(promedio 2018-2024)', fontweight='bold')
ax.set_xlabel('Pearson r (log10 pob vs IPC)')
for i, (v, n) in enumerate(zip(corr_reg_prom['r_mean'], corr_reg_prom['n'])):
    xp = v + 0.01 if v >= 0 else v - 0.01
    ha = 'left' if v >= 0 else 'right'
    ax.text(xp, i, f'{v:.3f}  (n~{int(n)})', va='center', ha=ha, fontsize=9)
plt.tight_layout()
plt.savefig(OUT / 'S7_corr_region.png', dpi=150, bbox_inches='tight')
plt.show()


## S9. Cuadrantes de inversión territorial con medianas propias

Clasifica los municipios en **cuatro cuadrantes** según si están por encima o debajo de las medianas de IPC y monto. A diferencia del análisis nacional, aquí **cada departamento y región usa sus PROPIAS medianas**.

**Rojo** = municipios con IPC <= mediana Y monto <= mediana de su propio territorio → **rezagados**.

### 9.1 Rezagados por departamento (medianas intra-departamentales)

In [ ]:
print("Calculando rezagados intra-departamentales...")
filas_rez_depto = []
for a in AÑOS:
    sub = df_mun[df_mun[COL_AÑO] == a].dropna(subset=[COL_IPC]).copy()
    for depto, grp in sub.groupby(COL_DEPTO):
        if len(grp) < 3: continue
        med_ipc   = grp[COL_IPC].median()
        med_monto = grp[COL_MONTO].median()
        n_rez = ((grp[COL_IPC] <= med_ipc) & (grp[COL_MONTO] <= med_monto)).sum()
        n_tot = len(grp)
        filas_rez_depto.append({
            'año': int(a), 'departamento': depto,
            'region': grp['region'].iloc[0] if 'region' in grp.columns else 'N/A',
            'n_rezagados': n_rez, 'n_total': n_tot,
            'pct_rezagados': round(n_rez / n_tot * 100, 1),
            'monto_total': grp[COL_MONTO].sum(),
        })

rez_depto = pd.DataFrame(filas_rez_depto)
rez_depto_prom = (rez_depto
    .groupby('departamento')
    .agg(pct_rez_mean = ('pct_rezagados', 'mean'),
         n_total = ('n_total', 'first'),
         region  = ('region', 'first'), monto_total = ('monto_total', 'sum'))
    .sort_values('pct_rez_mean', ascending=False)
    .reset_index())

print(f"\nRegistros departamento-año: {len(rez_depto)}")
print(f"Departamentos: {rez_depto['departamento'].nunique()}")

print("\nTop 5 deptos con MAYOR % de rezagados intra-departamentales:")
for _, r in rez_depto_prom.head(5).iterrows():
    print(f"  {r['departamento']:<20s}  {r['pct_rez_mean']:.1f}%  ({int(r['n_total'])} mun.)  [{r['region']}]")

print("\nTop 5 deptos con MENOR % de rezagados (mas balanceados):")
for _, r in rez_depto_prom.tail(5).iloc[::-1].iterrows():
    print(f"  {r['departamento']:<20s}  {r['pct_rez_mean']:.1f}%  ({int(r['n_total'])} mun.)  [{r['region']}]")

# Gráfica
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.suptitle('% de Municipios Rezagados por Departamento\n'
             '(medianas INTRA-departamentales, promedio 2018-2024)', fontweight='bold')
ax = axes[0]
top = rez_depto_prom.head(12).iloc[::-1]
ax.barh(range(len(top)), top['pct_rez_mean'],
        color=plt.cm.Reds(np.linspace(0.4, 0.9, len(top))), edgecolor='black')
ax.set_yticks(range(len(top))); ax.set_yticklabels(top['departamento'])
ax.set_xlim(0, 100); ax.axvline(50, color='black', ls='--')
ax.set_title('Mayor % de rezagados', fontweight='bold')
ax.set_xlabel('% municipios rezagados')
for i, (v, n) in enumerate(zip(top['pct_rez_mean'], top['n_total'])):
    ax.text(v + 1, i, f'{v:.1f}%  ({int(n)} mun.)', va='center', fontsize=8)
ax = axes[1]
bot = rez_depto_prom.tail(12).iloc[::-1]
ax.barh(range(len(bot)), bot['pct_rez_mean'],
        color=plt.cm.Greens(np.linspace(0.4, 0.9, len(bot))), edgecolor='black')
ax.set_yticks(range(len(bot))); ax.set_yticklabels(bot['departamento'])
ax.set_xlim(0, 100); ax.axvline(50, color='black', ls='--')
ax.set_title('Menor % de rezagados', fontweight='bold')
ax.set_xlabel('% municipios rezagados')
for i, (v, n) in enumerate(zip(bot['pct_rez_mean'], bot['n_total'])):
    ax.text(v + 1, i, f'{v:.1f}%  ({int(n)} mun.)', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(OUT / 'S9_rezagados_depto.png', dpi=150, bbox_inches='tight')
plt.show()


### 9.2 Rezagados por región (medianas intra-regionales)

In [ ]:
print("Calculando rezagados intra-regionales...")
filas_rez_reg = []
for a in AÑOS:
    sub = df_mun[df_mun[COL_AÑO] == a].dropna(subset=[COL_IPC]).copy()
    for region, grp in sub.groupby('region'):
        if pd.isna(region) or region == '' or len(grp) < 5: continue
        med_ipc   = grp[COL_IPC].median()
        med_monto = grp[COL_MONTO].median()
        n_rez = ((grp[COL_IPC] <= med_ipc) & (grp[COL_MONTO] <= med_monto)).sum()
        filas_rez_reg.append({
            'año': int(a), 'region': region,
            'n_rezagados': n_rez, 'n_total': len(grp),
            'pct_rezagados': round(n_rez / len(grp) * 100, 1),
        })

rez_region = pd.DataFrame(filas_rez_reg)
rez_reg_prom = (rez_region
    .groupby('region')
    .agg(pct_rez_mean = ('pct_rezagados', 'mean'), n_total = ('n_total', 'first'))
    .sort_values('pct_rez_mean', ascending=False)
    .reset_index())

print("\n% rezagados por region (promedio 2018-2024):")
for _, r in rez_reg_prom.iterrows():
    print(f"  {r['region']:<15s}  {r['pct_rez_mean']:.1f}%  ({int(r['n_total'])} mun.)")

# Gráfica
fig, ax = plt.subplots(figsize=(10, 5))
colores = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(rez_reg_prom)))
bars = ax.barh(range(len(rez_reg_prom)), rez_reg_prom['pct_rez_mean'],
               color=colores, edgecolor='black')
ax.set_yticks(range(len(rez_reg_prom)))
ax.set_yticklabels(rez_reg_prom['region']); ax.invert_yaxis()
ax.set_xlim(0, 100); ax.axvline(50, color='black', ls='--')
ax.set_title('% Municipios Rezagados por Region\n'
             '(medianas INTRA-regionales, promedio 2018-2024)', fontweight='bold')
ax.set_xlabel('% municipios rezagados')
for bar, v in zip(bars, rez_reg_prom['pct_rez_mean']):
    ax.text(v + 1, bar.get_y() + bar.get_height()/2, f'{v:.1f}%', va='center',
            fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'S9_rezagados_region.png', dpi=150, bbox_inches='tight')
plt.show()


## S13. % del presupuesto hacia municipios de baja IPC por territorio

¿Qué proporción del presupuesto **de cada territorio** llega a sus municipios con menor IPC?

Se parte cada departamento/región en dos mitades usando su **propia mediana de IPC** y se calcula qué % del monto total del territorio va a la mitad "Baja IPC".

**Referencia:** 50% = distribución equitativa entre ambas mitades.

### 13.1 % del presupuesto departamental hacia baja IPC

In [ ]:
print("Calculando % del monto hacia baja IPC por departamento...")
filas_pct_depto = []
for a in AÑOS:
    sub = df_mun[df_mun[COL_AÑO] == a].dropna(subset=[COL_IPC]).copy()
    for depto, grp in sub.groupby(COL_DEPTO):
        if len(grp) < 4: continue
        med = grp[COL_IPC].median()
        grp['grupo'] = np.where(grp[COL_IPC] <= med, 'Baja IPC', 'Alta IPC')
        agg = grp.groupby('grupo').agg(
            n = (COL_DIVIPOLA, 'count'), monto = (COL_MONTO, 'sum'))
        m_baja = agg.loc['Baja IPC', 'monto'] if 'Baja IPC' in agg.index else 0
        m_tot  = agg['monto'].sum()
        pct    = (m_baja / m_tot * 100) if m_tot > 0 else np.nan
        filas_pct_depto.append({
            'año': int(a), 'departamento': depto,
            'region': grp['region'].iloc[0] if 'region' in grp.columns else 'N/A',
            'n_baja': agg.loc['Baja IPC', 'n'] if 'Baja IPC' in agg.index else 0,
            'n_alta': agg.loc['Alta IPC', 'n'] if 'Alta IPC' in agg.index else 0,
            'monto_baja': m_baja, 'monto_total': m_tot,
            'pct_monto_baja_ipc': round(pct, 2),
        })

pct_depto = pd.DataFrame(filas_pct_depto)
pct_depto_prom = (pct_depto
    .groupby('departamento')
    .agg(pct_mean = ('pct_monto_baja_ipc', 'mean'),
         pct_std  = ('pct_monto_baja_ipc', 'std'),
         n_total  = ('n_baja', 'mean'),
         region   = ('region', 'first'), monto_total = ('monto_total', 'sum'))
    .sort_values('pct_mean')
    .reset_index())
pct_depto_prom['n_total'] = (pct_depto_prom['n_total'] * 2).astype(int)

print(f"\nRegistros departamento-año: {len(pct_depto)}")
print(f"Departamentos analizados: {pct_depto['departamento'].nunique()}")

print("\nTop 5 deptos con MENOR % hacia baja IPC (mas concentracion en alta IPC):")
for _, r in pct_depto_prom.head(5).iterrows():
    print(f"  {r['departamento']:<20s}  {r['pct_mean']:.1f}%  ({r['n_total']} mun.)  [{r['region']}]")

print("\nTop 5 deptos con MAYOR % hacia baja IPC (mas equitativos):")
for _, r in pct_depto_prom.tail(5).iloc[::-1].iterrows():
    print(f"  {r['departamento']:<20s}  {r['pct_mean']:.1f}%  ({r['n_total']} mun.)  [{r['region']}]")

# Gráfica
fig, ax = plt.subplots(figsize=(12, 16))
orden = pct_depto_prom['departamento'].tolist()
valores = pct_depto_prom['pct_mean'].tolist()
errores = pct_depto_prom['pct_std'].tolist()
colores = ['#E74C3C' if v < 15 else '#F39C12' if v < 25 else '#27AE60' for v in valores]
ax.barh(range(len(orden)), valores, color=colores, edgecolor='black',
        xerr=errores, capsize=2, alpha=0.85)
ax.set_yticks(range(len(orden))); ax.set_yticklabels(orden, fontsize=7); ax.invert_yaxis()
ax.axvline(50, color='black', ls='--', linewidth=1.2, label='50% (equidad perfecta)')
ax.set_xlim(0, 60)
ax.set_title('% del Presupuesto Departamental hacia Municipios de Baja IPC\n'
             '(medianas INTRA-departamentales, promedio 2018-2024)', fontweight='bold')
ax.set_xlabel('% del monto total del departamento')
ax.legend(loc='lower right')
for i, (v, n) in enumerate(zip(valores, pct_depto_prom['n_total'])):
    ax.text(v + 0.5, i, f'{v:.1f}%  ({n} mun.)', va='center', fontsize=7)
plt.tight_layout()
plt.savefig(OUT / 'S13_pct_baja_ipc_depto.png', dpi=150, bbox_inches='tight')
plt.show()


### 13.2 % del presupuesto regional hacia baja IPC

In [ ]:
print("Calculando % del monto hacia baja IPC por region...")
filas_pct_reg = []
for a in AÑOS:
    sub = df_mun[df_mun[COL_AÑO] == a].dropna(subset=[COL_IPC]).copy()
    for region, grp in sub.groupby('region'):
        if pd.isna(region) or region == '' or len(grp) < 5: continue
        med = grp[COL_IPC].median()
        grp['grupo'] = np.where(grp[COL_IPC] <= med, 'Baja IPC', 'Alta IPC')
        agg = grp.groupby('grupo').agg(
            n = (COL_DIVIPOLA, 'count'), monto = (COL_MONTO, 'sum'))
        m_baja = agg.loc['Baja IPC', 'monto'] if 'Baja IPC' in agg.index else 0
        m_tot  = agg['monto'].sum()
        pct    = (m_baja / m_tot * 100) if m_tot > 0 else np.nan
        filas_pct_reg.append({
            'año': int(a), 'region': region,
            'pct_monto_baja_ipc': round(pct, 2), 'n_total': len(grp),
        })

pct_region = pd.DataFrame(filas_pct_reg)
pct_reg_prom = (pct_region
    .groupby('region')
    .agg(pct_mean = ('pct_monto_baja_ipc', 'mean'),
         pct_std  = ('pct_monto_baja_ipc', 'std'),
         n_total  = ('n_total', 'first'))
    .sort_values('pct_mean')
    .reset_index())

print("\n% del presupuesto regional hacia baja IPC (promedio 2018-2024):")
for _, r in pct_reg_prom.iterrows():
    print(f"  {r['region']:<15s}  {r['pct_mean']:.1f}%  ({int(r['n_total'])} mun.)")

# Gráfica
fig, ax = plt.subplots(figsize=(10, 5))
colores = ['#E74C3C' if v < 15 else '#F39C12' if v < 25 else '#27AE60' for v in pct_reg_prom['pct_mean']]
ax.barh(range(len(pct_reg_prom)), pct_reg_prom['pct_mean'], color=colores,
        edgecolor='black', xerr=pct_reg_prom['pct_std'], capsize=3)
ax.set_yticks(range(len(pct_reg_prom)))
ax.set_yticklabels(pct_reg_prom['region']); ax.invert_yaxis()
ax.set_xlim(0, 60); ax.axvline(50, color='black', ls='--')
ax.set_title('% del Presupuesto Regional hacia Municipios de Baja IPC\n'
             '(medianas INTRA-regionales, promedio 2018-2024)', fontweight='bold')
ax.set_xlabel('% del monto total de la region')
for i, (v, n) in enumerate(zip(pct_reg_prom['pct_mean'], pct_reg_prom['n_total'])):
    ax.text(v + 0.5, i, f'{v:.1f}%  ({int(n)} mun.)', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUT / 'S13_pct_baja_ipc_region.png', dpi=150, bbox_inches='tight')
plt.show()


## Resumen ejecutivo territorial

In [ ]:
print('=' * 70)
print('RESUMEN TERRITORIAL CONSOLIDADO')
print('=' * 70)

print()
print('DATOS CARGADOS:')
print(f'  Municipios unicos:  {df_mun[COL_DIVIPOLA].nunique():,}')
print(f'  Departamentos:      {df_mun[COL_DEPTO].nunique()}')
print(f'  Regiones:           {df_mun["region"].nunique()}')

print()
print('S6 -- GINI IPC POR DEPARTAMENTO:')
print(f'  Mas desigual:       {gini_depto_prom.iloc[0]["departamento"]} (Gini={gini_depto_prom.iloc[0]["gini_ipc_mean"]:.3f})')
print(f'  Mas equitativo:     {gini_depto_prom.iloc[-1]["departamento"]} (Gini={gini_depto_prom.iloc[-1]["gini_ipc_mean"]:.3f})')

print()
print('S6 -- GINI IPC POR REGION:')
print(f'  Mas desigual:       {gini_reg_prom.iloc[0]["region"]} (Gini={gini_reg_prom.iloc[0]["gini_ipc_mean"]:.3f})')
print(f'  Mas equitativo:     {gini_reg_prom.iloc[-1]["region"]} (Gini={gini_reg_prom.iloc[-1]["gini_ipc_mean"]:.3f})')

print()
print('S7 -- CORRELACION POBLACION-IPC POR DEPARTAMENTO:')
print(f'  r mas negativo:     {corr_depto_prom.iloc[0]["departamento"]} (r={corr_depto_prom.iloc[0]["r_mean"]:.4f})')
print(f'  r mas positivo:     {corr_depto_prom.iloc[-1]["departamento"]} (r={corr_depto_prom.iloc[-1]["r_mean"]:.4f})')

print()
print('S7 -- CORRELACION POBLACION-IPC POR REGION:')
print(f'  r mas negativo:     {corr_reg_prom.iloc[0]["region"]} (r={corr_reg_prom.iloc[0]["r_mean"]:.4f})')
print(f'  r mas positivo:     {corr_reg_prom.iloc[-1]["region"]} (r={corr_reg_prom.iloc[-1]["r_mean"]:.4f})')

print()
print('S9 -- % REZAGADOS POR DEPARTAMENTO:')
print(f'  Mas rezagados:      {rez_depto_prom.iloc[0]["departamento"]} ({rez_depto_prom.iloc[0]["pct_rez_mean"]:.1f}%)')
print(f'  Menos rezagados:    {rez_depto_prom.iloc[-1]["departamento"]} ({rez_depto_prom.iloc[-1]["pct_rez_mean"]:.1f}%)')

print()
print('S9 -- % REZAGADOS POR REGION:')
print(f'  Mas rezagados:      {rez_reg_prom.iloc[0]["region"]} ({rez_reg_prom.iloc[0]["pct_rez_mean"]:.1f}%)')
print(f'  Menos rezagados:    {rez_reg_prom.iloc[-1]["region"]} ({rez_reg_prom.iloc[-1]["pct_rez_mean"]:.1f}%)')

print()
print('S13 -- % DEL PRESUPUESTO HACIA BAJA IPC POR DEPARTAMENTO:')
print(f'  Menor % (inequitativo): {pct_depto_prom.iloc[0]["departamento"]} ({pct_depto_prom.iloc[0]["pct_mean"]:.1f}%)')
print(f'  Mayor % (equitativo):   {pct_depto_prom.iloc[-1]["departamento"]} ({pct_depto_prom.iloc[-1]["pct_mean"]:.1f}%)')

print()
print('S13 -- % DEL PRESUPUESTO HACIA BAJA IPC POR REGION:')
print(f'  Menor % (inequitativo): {pct_reg_prom.iloc[0]["region"]} ({pct_reg_prom.iloc[0]["pct_mean"]:.1f}%)')
print(f'  Mayor % (equitativo):   {pct_reg_prom.iloc[-1]["region"]} ({pct_reg_prom.iloc[-1]["pct_mean"]:.1f}%)')

print()
print(f'Graficas guardadas en: {OUT}')
print()
print('Analisis territorial consolidado completado.')
